# dLuxShera ML Campaign Tracker

This notebook is a lightweight local tracker for compact ML inverse-model campaign artifacts already synchronized under `Results/hpc_imports/ml`. It does not SSH, rsync, train, or evaluate the frozen test set.

Use it to answer: what has run, what is complete, what is comparable, and roughly how well is it doing? Detailed scientific interpretation belongs in local study notebooks under `Results/ml_analysis/<study>/`.

Stable [study index and review status](README.md); completed scientific reviews: [S06](s06/review.md), [S07](s07/review.md), [S09](s09/review.md). Artifact completeness below is distinct from scientific review completion.

Per-parameter labels are recovered by matching each archived per-parameter RMSE to the column-wise RMSE recomputed from the saved arrays. `metrics.json` is written with sorted keys, so its key order is *not* the array column order; see the [study index](README.md).

## Setup and Filters

By default the tracker discovers every compact run under `Results/hpc_imports/ml`. Set the optional filters below when you want to narrow the view without editing analysis code.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "work" / "experiments" / "ml").exists():
            return candidate
    raise RuntimeError("Could not find the dLuxShera repository root.")


REPO_ROOT = find_repo_root()
ANALYSIS_DIR = REPO_ROOT / "work" / "experiments" / "ml" / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import campaign_analysis
import campaign_plots as cplots
campaign_analysis = importlib.reload(campaign_analysis)
cplots = importlib.reload(cplots)
from campaign_analysis import evaluation_contract_summary, experiment_summary_table, load_campaign, time_to_thresholds

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 80

RESULTS_ROOT = REPO_ROOT / "Results" / "hpc_imports" / "ml"
RESULT_ROOTS = [RESULTS_ROOT]

STUDIES = None          # e.g. ["S09"]
EXPERIMENTS = None      # e.g. ["S09-E01", "S09-E02"]
INCLUDE_RUNS = None     # e.g. ["S09-E01-R001"]
EXCLUDE_RUN_PATTERNS = ["SMOKE"]
RMSE_THRESHOLDS = [200, 150, 125, 100]

campaign = load_campaign(RESULT_ROOTS)
all_runs = campaign.runs.copy()

runs = all_runs.copy()
if not runs.empty:
    mask = pd.Series(True, index=runs.index)
    if STUDIES is not None:
        mask &= runs["study_id"].isin(STUDIES)
    if EXPERIMENTS is not None:
        mask &= runs["experiment_id"].isin(EXPERIMENTS)
    if INCLUDE_RUNS is not None:
        mask &= runs["run_id"].isin(INCLUDE_RUNS)
    for pattern in EXCLUDE_RUN_PATTERNS:
        mask &= ~runs["run_id"].astype(str).str.contains(pattern, case=False, na=False)
    runs = runs.loc[mask].copy()

selected_run_ids = set(runs["run_id"]) if not runs.empty else set()
history = campaign.history[campaign.history["run_id"].isin(selected_run_ids)].copy()
parameters = campaign.parameters[campaign.parameters["run_id"].isin(selected_run_ids)].copy()
slices = campaign.slices[campaign.slices["run_id"].isin(selected_run_ids)].copy()
distance_bins = campaign.distance_bins[campaign.distance_bins["run_id"].isin(selected_run_ids)].copy()

display(Markdown(f"**Local result root:** `{RESULTS_ROOT}`"))
display(Markdown(f"**Selected runs:** {len(runs)} of {len(all_runs)} discovered."))

## Local Mirror and Completion

This section surfaces what is present locally and whether the compact artifact set is complete. It is a local mirror view, not a remote sync operation.

In [ ]:
if campaign.sync_manifests.empty:
    display(Markdown("No `sync_manifest.json` files were found under the selected result roots."))
else:
    sync_cols = [c for c in ["site", "synced_at_utc", "remote_root", "local_root", "include_checkpoints", "include_logs"] if c in campaign.sync_manifests]
    display(campaign.sync_manifests[sync_cols].sort_values(sync_cols[1] if len(sync_cols) > 1 else sync_cols[0], ascending=False))

if runs.empty:
    display(Markdown("No runs matched the current filters."))
else:
    discovered = (
        runs.groupby(["site", "study_id", "status"], dropna=False)
        .size()
        .rename("run_count")
        .reset_index()
        .sort_values(["study_id", "site", "status"])
    )
    display(discovered)
    sites = sorted(str(value) for value in runs["site"].dropna().unique()) if "site" in runs else []
    studies = sorted(str(value) for value in runs["study_id"].dropna().unique()) if "study_id" in runs else []
    display(Markdown(f"Discovered sites: `{sites}`. Selected studies: `{studies}`."))

for message in campaign.warnings:
    display(Markdown(f"**Loader warning:** {message}"))

### Stopping Reason and Budget

`status == "complete"` means the compact artifact set is present. It says nothing
about whether the epoch budget was reached. A run that stops before `max_epochs`
with `early_stopped == True` is a legitimately finished run, not a failed or
truncated one. The table below separates those cases so neither is misread, and
keeps `best_epoch` next to `epochs_completed` so best-checkpoint metrics are not
confused with final-epoch values.

In [ ]:
if runs.empty:
    display(Markdown("No runs matched the current filters."))
else:
    def stopping_reason(row: pd.Series) -> str:
        if bool(row.get("reached_max_epochs")):
            return "reached budget"
        if bool(row.get("early_stopped")):
            return "early stopped (legitimate)"
        return "stopped before budget, not flagged early-stopped"

    stopping = runs.assign(stopping_reason=runs.apply(stopping_reason, axis=1))
    stopping_cols = [
        "study_id", "experiment_id", "run_id", "seed", "status", "stopping_reason",
        "epochs_completed", "max_epochs", "best_epoch",
    ]
    display(
        stopping[[c for c in stopping_cols if c in stopping]]
        .sort_values(["study_id", "experiment_id", "run_id"], na_position="last")
    )
    display(
        stopping.groupby(["study_id", "stopping_reason"], dropna=False)
        .size()
        .rename("run_count")
        .reset_index()
    )
    unexplained = stopping["stopping_reason"].eq(
        "stopped before budget, not flagged early-stopped"
    )
    if unexplained.any():
        display(Markdown(
            "**Warning:** runs stopped short of the budget without an early-stopping "
            f"flag: `{sorted(stopping.loc[unexplained, 'run_id'])}`. Check whether "
            "these are in-flight or interrupted snapshots before using them."
        ))

    dirty = []
    for run_id, record in campaign.run_records.items():
        git = record.manifest.get("git") or {}
        if git.get("dirty"):
            dirty.append(run_id)
    if dirty:
        display(Markdown(
            f"**Provenance note:** {len(dirty)} selected runs record a dirty source "
            "tree. The exact working-tree diff is not recoverable from the compact "
            "artifacts; record this limitation in the study review."
        ))

## Run Overview and Evaluation Contracts

The run table keeps identity and headline metrics together: experiment, seed, completion, epochs, runtime, global Fisher metrics, and binary-separation RMSE in milliarcseconds when available.

In [ ]:
overview_cols = [
    "study_id", "site", "experiment_id", "run_id", "seed", "status",
    "epochs_completed", "best_epoch", "best_fisher_rmse", "zero_baseline_fisher_rmse",
    "mse_skill", "rmse_reduction", "separation_rmse_mas", "total_training_seconds",
    "source_commit", "validation_manifest_sha256", "test_evaluated",
]

if runs.empty:
    display(Markdown("No run overview available."))
else:
    overview = runs[[c for c in overview_cols if c in runs.columns]].copy()
    if "total_training_seconds" in overview:
        overview["runtime_hr"] = (pd.to_numeric(overview["total_training_seconds"], errors="coerce") / 3600.0).round(2)
        overview = overview.drop(columns=["total_training_seconds"])
    for col in ["best_fisher_rmse", "zero_baseline_fisher_rmse", "separation_rmse_mas"]:
        if col in overview:
            overview[col] = pd.to_numeric(overview[col], errors="coerce").round(3)
    for col in ["mse_skill", "rmse_reduction"]:
        if col in overview:
            overview[col] = pd.to_numeric(overview[col], errors="coerce").round(4)
    display(overview.sort_values(["study_id", "experiment_id", "seed"], na_position="last"))

contract = evaluation_contract_summary(runs)
if not contract.empty:
    display(Markdown("### Evaluation Contract Summary"))
    contract_cols = [
        "study_id", "experiment_id", "run_id", "prepared_dataset_hash",
        "split_registry_content_sha256", "validation_manifest_sha256", "test_evaluated",
    ]
    display(contract[[c for c in contract_cols if c in contract]].sort_values(["study_id", "experiment_id", "run_id"]))
    identity_cols = ["prepared_dataset_hash", "split_registry_content_sha256", "validation_manifest_sha256"]
    differing = [col for col in identity_cols if col in contract and contract[col].dropna().nunique() > 1]
    if differing:
        display(Markdown("**Warning:** selected runs do not share a single evaluation contract. Compare within matching study/manifest groups unless you have a specific cross-study reason."))
    if "test_evaluated" in contract and contract["test_evaluated"].fillna(False).any():
        display(Markdown("**Warning:** at least one selected run has `test_evaluated == True`."))

## Experiment-Level Headline Results

These summaries aggregate by experiment while retaining seed scatter in the plots. They are meant for quick triage, not final scientific interpretation.

In [ ]:
summary = experiment_summary_table(runs)
if summary.empty:
    display(Markdown("No experiment summary available."))
else:
    summary_cols = [
        "study_id", "experiment_id", "run_count", "complete_count", "seed_count",
        "best_fisher_rmse_mean", "best_fisher_rmse_std",
        "mse_skill_mean", "rmse_reduction_mean",
        "separation_rmse_mas_mean", "separation_rmse_mas_std",
        "total_training_seconds_mean", "best_epoch_mean",
    ]
    compact = summary[[c for c in summary_cols if c in summary]].copy()
    if "total_training_seconds_mean" in compact:
        compact["runtime_hr_mean"] = (compact["total_training_seconds_mean"] / 3600.0).round(2)
        compact = compact.drop(columns=["total_training_seconds_mean"])
    for col in compact.select_dtypes(include=["float", "float64", "float32"]).columns:
        compact[col] = compact[col].round(3)
    display(compact.sort_values(["study_id", "experiment_id"]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    cplots.plot_experiment_metric_summary(runs, "best_fisher_rmse", ax=axes[0], ylabel="Best validation Fisher RMSE")
    cplots.add_common_zero_baseline(axes[0], runs)
    cplots.plot_experiment_metric_summary(runs, "separation_rmse_mas", ax=axes[1], ylabel="Separation RMSE (mas)")
    plt.show()

## Training Progress

These curves are frozen-validation metrics from the compact history files. They are useful for campaign triage but do not identify which physical parameter improved at a given epoch unless the artifacts explicitly preserve that history.

In [ ]:
if history.empty:
    display(Markdown("No training-history rows are available for the selected runs."))
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    cplots.plot_validation_metric_by_experiment(history, metric="validation_overall_rmse", x="epoch", ax=axes[0], ylabel="Validation Fisher RMSE")
    cplots.plot_validation_metric_by_experiment(history, metric="best_validation_rmse_so_far", x="epoch", ax=axes[1], ylabel="Best-so-far validation Fisher RMSE")
    plt.show()

    thresholds = time_to_thresholds(history, RMSE_THRESHOLDS)
    if not thresholds.empty:
        threshold_view = thresholds.merge(runs[["run_id", "study_id", "experiment_id", "seed"]], on="run_id", how="left", suffixes=("", "_run"))
        display(threshold_view.sort_values(["threshold", "study_id", "experiment_id", "seed"]))

## Science-Priority Physical Metrics

Physical metrics are shown per parameter only where the artifacts preserve meaningful native units. Separation is displayed in milliarcseconds; mixed-unit parameters are not collapsed into a single physical score.

In [ ]:
if parameters.empty or "parameter" not in parameters:
    display(Markdown("No per-parameter metric table is available."))
else:
    separation = parameters[parameters["parameter"].eq("source.separation_as")].copy()
    if separation.empty:
        display(Markdown("No binary-separation physical metric was found in the selected artifacts."))
    else:
        sep_cols = [
            "study_id", "experiment_id", "run_id", "seed",
            "fisher_model_rmse", "fisher_baseline_rmse", "fisher_mse_skill",
            "physical_rmse_display", "physical_mae_display", "physical_display_unit",
        ]
        sep_view = separation[[c for c in sep_cols if c in separation]].copy()
        for col in ["fisher_model_rmse", "fisher_baseline_rmse", "fisher_mse_skill", "physical_rmse_display", "physical_mae_display"]:
            if col in sep_view:
                sep_view[col] = pd.to_numeric(sep_view[col], errors="coerce").round(3)
        display(sep_view.sort_values(["study_id", "experiment_id", "seed"]))
        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        cplots.plot_parameter_physical_rmse_by_experiment(parameters, parameter="source.separation_as", ax=ax)
        plt.show()